# Informe de selección de modelo: Resultados Saber 11 (Antioquia, 2022-2)

**Grupo**: Jesús Emilio Palacio Agudelo [1152700501] - Sebastian Zuluaga Ruiz [1017222512] - Juan Felipe Muñoz Zuluaga [1152454728]
- **Variante**: regresión
- **Dataset**: [Resultados únicos Saber 11; https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe/about_data]
- **Fecha**: [2026-11-06]

## 0. Preparación

La celda de abajo importa todo lo que usa el notebook y fija la semilla. Si corre sin error, el entorno está listo:

In [25]:
pip install optuna

In [1]:
import numpy as np
import optuna
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
import requests
from xgboost import XGBRegressor
from sklearn.base import BaseEstimator, RegressorMixin

pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42

## 1. Los datos

La celda de abajo descarga de la API los resultados del periodo 20224 en Antioquia y muestra su tamaño:

In [2]:
BASE_URL = "https://www.datos.gov.co/resource/kgxf-xxbe.json"
FILTRO = "periodo = '20224' AND cole_depto_ubicacion = 'ANTIOQUIA'"

resp = requests.get(
    BASE_URL,
    params={"$where": FILTRO, "$limit": 200_000, "$order": "estu_consecutivo"},
# El $order garantiza que los datos lleguen siempre en la misma fila, para que la semilla, que sí es aleatoria pero controlada, produzca siempre la misma partición.
    timeout=300,
)
resp.raise_for_status()

raw = pd.DataFrame(resp.json())
print(f"{raw.shape[0]:,} filas x {raw.shape[1]} columnas")

144,766 filas x 51 columnas


El dataset elegido contiene información de las pruebas saber 11 de los colegios a nivel Nacional, por lo que cuenta con aproximadamente 7 millones de filas. Esto nos llevó a tomar la decisión de limitarlo tanto por período de la información como por el departamento de modo que pudieramos contar igual con una buena cantidad de información que nos ayudara a analizar la variable objetivo bajo la ejecución de varios modelos y donde la validación cruzada pudiese ser eficiente.

La celda de abajo detecta duplicados. Error en la fuente de datos:

In [3]:
print(f"filas en raw: {len(raw):,}")
print(f"filas repetidas: {raw.duplicated().sum():,}")

filas en raw: 144,766
filas repetidas: 72,383


Hemos identificado que la mitad de las filas se encuentran duplicadas, por lo que las eliminaremos de modo que sólo contemos con valores únicos que puedan aportar valor a nuestro dataset.

La siguiente celda elimina los duplicados y valida valores únicos en la columna estu_consecutivo:

In [4]:
raw = raw.drop_duplicates().reset_index(drop=True)
print(f"filas: {len(raw):,}")
print(f"identificadores únicos: {raw['estu_consecutivo'].nunique():,}")

filas: 72,383
identificadores únicos: 72,383


La siguiente celda muestra las primeras filas:

In [5]:
raw.head()

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,cole_genero,cole_jornada,...,fami_educacionpadre,fami_estratovivienda,fami_tieneinternet,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global,fami_cuartoshogar,fami_personashogar,fami_tieneautomovil,fami_tienecomputador,fami_tienelavadora
0,20224,CC,SB11202240000144,URBANO,N,A,TÉCNICO/ACADÉMICO,105040000212,105040000212,05,05040,753228,ANTIOQUIA,MIXTO,SABATINA,...,Secundaria (Bachillerato) completa,Estrato 1,No,A-,38,27,35,36,34,167,NaN,NaN,NaN,NaN,NaN
1,20224,CC,SB11202240000145,URBANO,N,A,TÉCNICO/ACADÉMICO,105040000212,105040000212,05,05040,753228,ANTIOQUIA,MIXTO,SABATINA,...,Secundaria (Bachillerato) incompleta,Estrato 1,No,A-,31,33,24,36,32,156,NaN,NaN,NaN,NaN,NaN
2,20224,CC,SB11202240000147,URBANO,N,A,TÉCNICO/ACADÉMICO,105040000212,105040000212,05,05040,753228,ANTIOQUIA,MIXTO,SABATINA,...,Primaria incompleta,Sin Estrato,No,A-,37,39,31,32,32,169,NaN,NaN,NaN,NaN,NaN
3,20224,CC,SB11202240000148,URBANO,N,A,TÉCNICO/ACADÉMICO,105040000212,105040000212,05,05040,753228,ANTIOQUIA,MIXTO,SABATINA,...,NaN,NaN,NaN,A-,37,31,28,31,31,154,NaN,NaN,NaN,NaN,NaN
4,20224,CC,SB11202240000149,URBANO,N,A,TÉCNICO/ACADÉMICO,105040000212,105040000212,05,05040,753228,ANTIOQUIA,MIXTO,SABATINA,...,NaN,NaN,NaN,A-,42,29,34,31,35,165,NaN,NaN,NaN,NaN,NaN


La siguiente celda muestra el tipo, los faltantes y los valores distintos de cada columna. Busquen
columnas con fuga, códigos guardados como números y columnas casi vacías:

In [6]:
pd.DataFrame(
    {
        "tipo": raw.dtypes.astype(str),
        "faltantes": raw.isna().sum(),
        "valores distintos": raw.nunique(),
    }
)

,tipo,faltantes,valores distintos
periodo,str,0,1
estu_tipodocumento,str,0,9
estu_consecutivo,str,0,72383
cole_area_ubicacion,str,0,2
cole_bilingue,str,11861,2
cole_calendario,str,0,2
cole_caracter,str,2372,4
cole_cod_dane_establecimiento,str,0,1122
cole_cod_dane_sede,str,0,1447
cole_cod_depto_ubicacion,str,0,1


Gracias a esta descripcion, pudimos notar que el dataset original, al parecer les tiene el mismo formato a todas sus columnas, por lo que será un ajuste que tendremos que corregir, además por variables como cole_nombre_sede tambien se puede apreciar que contamos con información de más de 1,000 colegios del departamento de Antioquia.

La siguiente celda busca marcas ocultas de faltantes en todas las columnas de texto:

In [ ]:
SOSPECHOSOS = ["-", "?", "", " ", "NA", "N/A", "NO APLICA", "NO SABE", "SIN ESTRATO", "NULL", "NONE"]

text_cols = raw.select_dtypes(include="str").columns
hallazgos = {
    col: raw[col].astype(str).str.strip().str.upper().isin(SOSPECHOSOS).sum()
    for col in text_cols
}
pd.Series(hallazgos)[lambda s: s > 0].sort_values(ascending=False)

fami_educacionpadre     7221
fami_educacionmadre     1851
cole_caracter           1817
fami_estratovivienda    1561
dtype: int64

La siguiente celda cuenta las categorías de las cuatro columnas de texto sospechosas, para buscar una marca oculta de faltante.

En `fami_educacionpadre` y `fami_educacionmadre`, el valor `No sabe` no informa
  el nivel educativo, así que pasa a faltante en los arreglos de estructura de esta sección:

In [9]:
COLS_SOSPECHOSAS = [
    "fami_educacionpadre",
    "fami_educacionmadre",
    "cole_caracter",
    "fami_estratovivienda",
]

print(f"filas en raw: {len(raw):,}\n")
for col in COLS_SOSPECHOSAS:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False))
    print()

filas en raw: 72,383

--- fami_educacionpadre ---
fami_educacionpadre
Secundaria (Bachillerato) completa      16156
Primaria incompleta                     12596
Secundaria (Bachillerato) incompleta     9711
No sabe                                  5966
Educación profesional completa           5710
Primaria completa                        5575
Técnica o tecnológica completa           4844
NaN                                      4267
Ninguno                                  2500
Técnica o tecnológica incompleta         1371
No Aplica                                1255
Postgrado                                1240
Educación profesional incompleta         1192
Name: count, dtype: int64

--- fami_educacionmadre ---
fami_educacionmadre
Secundaria (Bachillerato) completa      19927
Secundaria (Bachillerato) incompleta     9206
Primaria incompleta                      8917
Técnica o tecnológica completa           8348
Educación profesional completa           7605
Primaria completa          

Gracias a la busqueda de columnas que nombramos como sospechosas, encontramos que existen valores que son necesarios de imputar para de este modo mejorar el dataset y garantizar un mejor resultados en los modelos que pondremos a competir.

Es importante mencionar que dentro de lo evidenciado, hemos determinado que será importante imputar las columnas "No sabe" y "NaN" debido a que ellas no aportan valor al dataset, como quizás si lo podrían hacer las columnas "Ninguno" y "No aplica", teniendo en cuenta que cuando se refieren a "Ninguno" podría tratarse de padres o madres que no tuvieron ningun tipo de educación y para el caso de "No aplica" podría tratarse de padres o madres que fallecieron o presentaron alguna otra eventualidad que no permite medirla o representarla mediante esta pregunta concreta.

La siguiente celda muestra el valor más frecuente de las columnas que casi no cambian:

In [10]:
for col in ["estu_nacionalidad", "estu_pais_reside", "estu_privado_libertad"]:
    print(raw[col].value_counts(normalize=True).round(4).head(2), "\n")

estu_nacionalidad
COLOMBIA     0.9850
VENEZUELA    0.0141
Name: proportion, dtype: float64 

estu_pais_reside
COLOMBIA     0.9850
VENEZUELA    0.0141
Name: proportion, dtype: float64 

estu_privado_libertad
N    0.9985
S    0.0015
Name: proportion, dtype: float64 



La siguiente celda hace los arreglos de estructura: quita las columnas que no se usan, convierte el objetivo a número y crea las predictoras numéricas. Cómo rellenar los faltantes lo decide el pipeline de la sección 4. Al final muestra los faltantes que quedan en cada columna:

In [11]:
TARGET = "punt_global"  # ✏️ su objetivo

# ✏️ las columnas que quitan; la razón de cada una va en el entregable 1
DROPPED_COLS = [
    # fuga: se conocen con los resultados del examen
    "punt_lectura_critica",
    "punt_matematicas",
    "punt_c_naturales",
    "punt_sociales_ciudadanas",
    "punt_ingles",
    "desemp_ingles",
    "estu_estadoinvestigacion",
    # un solo valor por los filtros de descarga
    "periodo",
    "cole_depto_ubicacion",
    "cole_cod_depto_ubicacion",
    "estu_estudiante",
    # identificadores
    "estu_consecutivo",
    "cole_codigo_icfes",
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede",
    # demasiadas categorías
    "cole_nombre_establecimiento",
    "cole_nombre_sede",
    # códigos que repiten un nombre
    "cole_cod_mcpio_ubicacion",
    "estu_cod_reside_depto",
    "estu_cod_reside_mcpio",
    "estu_cod_depto_presentacion",
    "estu_cod_mcpio_presentacion",
    # repiten otra columna
    "estu_depto_reside",
    "estu_mcpio_reside",
    "estu_depto_presentacion",
    "estu_mcpio_presentacion",
    "estu_pais_reside",
    # casi siempre el mismo valor
    "estu_nacionalidad",
    "estu_privado_libertad",
    # se reemplaza por edad
    "estu_fechanacimiento",
]

data = raw.copy()

# ✏️ sus arreglos de estructura
# 1. el objetivo pasa de texto a número
data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")

# 2. edad en años cumplidos a la fecha aproximada del examen (periodo 2022-2)
FECHA_EXAMEN = pd.Timestamp("2022-08-01")
nacimiento = pd.to_datetime(data["estu_fechanacimiento"], format="%d/%m/%Y", errors="coerce")
data["edad"] = np.floor((FECHA_EXAMEN - nacimiento).dt.days / 365.25)
data.loc[~data["edad"].between(10, 80), "edad"] = np.nan  # edades imposibles pasan a faltante

# 3. cuartos del hogar: de palabra a número
CUARTOS = {"Uno": 1, "Dos": 2, "Tres": 3, "Cuatro": 4, "Cinco": 5, "Seis o mas": 6}
data["fami_cuartoshogar"] = data["fami_cuartoshogar"].map(CUARTOS)

# 4. personas del hogar: de rango a su punto medio
PERSONAS = {"1 a 2": 1.5, "3 a 4": 3.5, "5 a 6": 5.5, "7 a 8": 7.5, "9 o más": 9}
data["fami_personashogar"] = data["fami_personashogar"].map(PERSONAS)

# 5. "No sabe" en la educación de los padres pasa a faltante: no informa el nivel educativo
for col in ["fami_educacionpadre", "fami_educacionmadre"]:
    data[col] = data[col].replace("No sabe", np.nan)

data = data.drop(columns=DROPPED_COLS)

print(f"{data.shape[0]:,} filas x {data.shape[1]} columnas")
data.isna().sum()

72,383 filas x 22 columnas


estu_tipodocumento          0
cole_area_ubicacion         0
cole_bilingue           11861
cole_calendario             0
cole_caracter            2372
cole_genero                 0
cole_jornada                0
cole_mcpio_ubicacion        0
cole_naturaleza             0
cole_sede_principal         0
estu_genero                 0
fami_educacionmadre      5891
fami_educacionpadre     10233
fami_estratovivienda     4722
fami_tieneinternet       4257
punt_global                 0
fami_cuartoshogar        3411
fami_personashogar       3304
fami_tieneautomovil      3606
fami_tienecomputador     3413
fami_tienelavadora       3424
edad                       14
dtype: int64

La siguiente celda resume el objetivo: su media, su dispersión, su mínimo, su máximo y sus cuartiles:

In [12]:
data[TARGET].describe().round(1)

count    72383.0
mean       245.9
std         51.6
min        100.0
25%        205.0
50%        242.0
75%        283.0
max        477.0
Name: punt_global, dtype: float64

La siguiente celda resume el objetivo: su mediana y valores únicos:

In [13]:
print(f"mediana: {data[TARGET].median()}")
print(f"valores distintos: {data[TARGET].nunique()}")

mediana: 242.0
valores distintos: 315


✍️ **Entregable 1 de 8: ficha del dataset**

- **Nombre y fuente**: *Resultados únicos Saber 11*, del ICFES, publicado en el
  [Portal de Datos Abiertos de Colombia](https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe)
  y descargado por su API. De acuerdo a nuestra necesidad y el filtro estrategico aplicado, el archivo guarda solo el periodo 20224 (2022-2, calendario A) y los
  colegios pertenecientes al departamento de Antioquia: cuenta con 144,766 filas, de las cuales 72,383 eran duplicados exactos de la fuente.
  Sin duplicados quedan 72,383 estudiantes.
- **Qué representa una fila**: un estudiante que presentó el examen: su género y edad, cómo es su
  colegio (naturaleza, jornada, carácter, zona, municipio) y cómo es su hogar (estrato, educación de
  los padres, personas, cuartos y bienes).
- **Tamaño después de los arreglos**: 72,383 filas × 22 columnas (21 predictoras y el
  objetivo).
- **El objetivo**: `punt_global`, el puntaje global del examen, el dataset lo proporciona en puntos, especificamente en una escala de 0 a 500. Los puntajes reales obtenidos por los estudiantes, se encuentra en un rango de
  100 a 477 puntos, con una media de 245,9 y una mediana de 242, donde evidenciamos 
  315 valores distintos.
- **La decisión**: Predecir el puntaje de un estudiante perteneciente a un colegio del departamente previo a la presentación de su examen, para que de este modo, la Secretaría de
  Educación de Antioquia focalice los programas de refuerzo en los estudiantes y colegios con mayor
  riesgo de bajo desempeño.
- **Columnas quitadas**, por grupo:
  - **con fuga**: `punt_lectura_critica`, `punt_matematicas`, `punt_c_naturales`,
    `punt_sociales_ciudadanas` y `punt_ingles` (se conocen con los resultados, y el puntaje global se
    calcula con ellos), `desemp_ingles` (se asigna con el puntaje de inglés) y
    `estu_estadoinvestigacion` (solo existe cuando ya hay resultado). Ninguna se conoce antes del
    examen y por ello no son relevantes para los modelos a analizar;
  - **un solo valor**: `periodo`, `cole_depto_ubicacion`, `cole_cod_depto_ubicacion` y
    `estu_estudiante`, debido al filtro estrategico de la descarga, todos tienen el mismo período 2022-4 que al no tener variación, no representan información útil;
  - **identificadores**: `estu_consecutivo`, `cole_codigo_icfes`, `cole_cod_dane_establecimiento` y
    `cole_cod_dane_sede`, estos son identificadores que sirven a entidades estatales, pero en un proceso de train, podrían ocasionar un sobreajuste donde los modelos podrían memorizarlos;
  - **demasiadas categorías**: `cole_nombre_establecimiento` y `cole_nombre_sede`, al registrar más de 1,000 al aplicar preprocesamiento, se podrían crear miles de columnas dispersas que ralentizarian el entrenamiento, esto debido a que el colegio ya queda descrito por su naturaleza, jornada, carácter, zona y municipio;
  - **códigos que repiten un nombre**: `cole_cod_mcpio_ubicacion`, `estu_cod_reside_depto`,
    `estu_cod_reside_mcpio`, `estu_cod_depto_presentacion` y `estu_cod_mcpio_presentacion`;
  - **repiten otra columna**: `estu_depto_reside`, `estu_mcpio_reside`, `estu_depto_presentacion` y
    `estu_mcpio_presentacion` repiten el municipio del colegio; `estu_pais_reside` repite
    `estu_nacionalidad` En este caso, tanto el municipio como el departamento siempre serán los mismos debido al filtro aplicado inicialmente y por ello lo mejor en este caso, es eliminar la redundancia;
  - **casi siempre el mismo valor**: `estu_nacionalidad` (98.5 % Colombia) y
    `estu_privado_libertad` (99.85 % "N") En estos casos, la población está muy concentrada en una nacionalidad y al presentarse una minoria tan poco representativa, los modelos no tendría suficiente información para darle un peso o importancia real;
  - **se reemplaza por otra**: `estu_fechanacimiento` pasa a ser `edad` En busca de darle mayor poder a los modelos y al ser este un problema de regresión, es importante efectuar este cambio en busca de obtener mejores resultados.
- **Arreglos antes de la partición**: se quitan los duplicados exactos, para que un estudiante no
  quede en entrenamiento y en prueba a la vez, evitando así la contaminación cruzada; `punt_global` pasa de texto a número; `edad` se
  calcula a la fecha aproximada del examen (agosto de 2022), y las edades biologicamente imposibles pasan a ser
  faltantes; `fami_cuartoshogar` pasa de palabra a número; `fami_personashogar` pasa de rango a su
  punto medio numérico; `No sabe` en la educación del padre y de la madre pasa a faltante, porque no informa el
nivel educativo. `No Aplica`, `Ninguno` y `Sin Estrato` se conservan como categorías: son respuestas
reales, no faltantes. Ningún arreglo usa información de otras filas.
- **Requisitos de la actividad**: El dataset final consolidado cuenta con 72,383 filas; la variable objetivo cuenta con 315 valores distintos; Se dispone de
  21 variables predictoras, 3 numéricas y 18 de texto; faltantes en `cole_bilingue`, `cole_caracter`, `edad` y las columnas `fami_*`, y columnas categóricas que preparar; cumpliendo así con todos los requisitos propuestos.

## 2. La métrica

La celda de abajo guarda el nombre de la métrica en una constante. Todos los pasos siguientes la usan:

In [14]:
SCORING = "neg_root_mean_squared_error"

✍️ **Entregable 2 de 8: la métrica**

- **La métrica**: Utilizaremos el RMSE en puntos del examen Saber 11, en el torneo, el ajuste y en la prueba final.
- **Su unidad**: puntos, en la escala de 0 a 500 del puntaje global. El puntaje mediano es 242
  puntos: un RMSE de 40 puntos sería un error típico de cerca del 17 % de ese puntaje.
- **Por qué los errores grandes cuentan más**: si el modelo estima 290 puntos para un estudiante
  que en realidad saca 200, ese estudiante queda fuera del programa de refuerzo justo cuando más lo
  necesita; muchos errores de 5 puntos casi no cambian a quién se focaliza. Unos pocos errores
  grandes hacen más daño que muchos pequeños.

## 3. La partición

La celda de abajo separa las predictoras del objetivo y aparta el 20 % de las filas como conjunto de prueba:

In [15]:
X = data.drop(columns=TARGET)
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

La siguiente celda muestra el tamaño de cada parte y la media de su objetivo:

In [16]:
pd.DataFrame(
    {
        "filas": [len(y_train), len(y_test)],
        "media del objetivo": [y_train.mean(), y_test.mean()],
    },
    index=["entrenamiento", "prueba"],
).round(1)

,filas,media del objetivo
entrenamiento,57906,245.8
prueba,14477,246.3


La celda de abajo crea los cinco folds y calcula con cuántas filas se entrena y se valida un modelo
en cada fold:

In [17]:
N_FOLDS = 5
cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fit_rows = len(X_train) * (N_FOLDS - 1) // N_FOLDS
print(f"filas para entrenar en cada fold: {fit_rows:,}")
print(f"filas para validar en cada fold: {len(X_train) - fit_rows:,}")

filas para entrenar en cada fold: 46,324
filas para validar en cada fold: 11,582


✍️ **Entregable 3 de 8: la partición**

- **Tamaños**: • El conjunto de datos neto se dividió en 57,906 observaciones para el entrenamiento y 14,477 observaciones para la prueba final.
- **Por qué esa proporción para la prueba**: La partición del 20 % para el conjunto de prueba es óptima para este volumen de datos. Disponer de 14,477 estudiantes en la evaluación final garantiza una muestra lo suficientemente grande para que la métrica de generalización (RMSE) sea estadísticamente estable y no dependa del azar.
- **Número de folds, y filas para entrenar y para validar en un fold**: Se configuró una estrategia de validación cruzada de 5 folds (K-Fold) con barajado aleatorio sobre el conjunto de entrenamiento. En cada una de las 5 iteraciones, los modelos se entrenan con 46,324 registros y se validan con 11,582 registros, permitiendo una estimación robusta del desempeño sin tocar los datos de prueba..
- **Qué tan parecido es el objetivo en las dos partes**: Las dos divisiones principales muestran una distribución altamente consistente. El puntaje global promedio es de 245.8 puntos en el conjunto de entrenamiento y 246.3 puntos en el de prueba. Esta mínima diferencia de 0.5 puntos resulta insignificante si se contrasta con la desviación estándar global del examen (51.6 puntos), lo que demuestra que la partición aleatoria controlada segmentó de manera homogénea las poblaciones estudiantiles.
- **La regla que siguen con el conjunto de prueba desde ahora**: Como garantía estricta contra el sobreajuste y la fuga de datos (data leakage), las matrices X_test y y_test permanecen completamente aisladas de los procesos de escalamiento, imputación, torneo de modelos y ajuste de hiperparámetros. Se utilizarán por única vez en la Sección 7 como una auditoría ciega del modelo final.

## 4. El pipeline de preprocesamiento

La siguiente tabla clasifica las 18 predictoras categóricas. Las ordinales tienen un orden natural
entre sus categorías y se codifican con su posición; las nominales no tienen orden y se codifican
con one-hot:

| # | Variable | Qué es | Tipo | Codificación |
|---|---|---|---|---|
| 1 | `fami_estratovivienda` | estrato de la vivienda | Ordinal | `OrdinalEncoder` |
| 2 | `fami_educacionpadre` | nivel educativo del padre | Ordinal | `OrdinalEncoder` |
| 3 | `fami_educacionmadre` | nivel educativo de la madre | Ordinal | `OrdinalEncoder` |
| 4 | `estu_tipodocumento` | tipo de documento (TI, CC, CE…) | Nominal | `OneHotEncoder` |
| 5 | `estu_genero` | género del estudiante | Nominal (binaria) | `OneHotEncoder` |
| 6 | `cole_area_ubicacion` | zona urbana o rural | Nominal (binaria) | `OneHotEncoder` |
| 7 | `cole_bilingue` | colegio bilingüe (S/N) | Nominal (binaria) | `OneHotEncoder` |
| 8 | `cole_calendario` | calendario A o B | Nominal (binaria) | `OneHotEncoder` |
| 9 | `cole_caracter` | académico, técnico o técnico/académico | Nominal | `OneHotEncoder` |
| 10 | `cole_genero` | mixto, femenino o masculino | Nominal | `OneHotEncoder` |
| 11 | `cole_jornada` | mañana, tarde, completa, única… | Nominal | `OneHotEncoder` |
| 12 | `cole_mcpio_ubicacion` | municipio del colegio | Nominal | `OneHotEncoder` |
| 13 | `cole_naturaleza` | oficial o no oficial | Nominal (binaria) | `OneHotEncoder` |
| 14 | `cole_sede_principal` | sede principal (S/N) | Nominal (binaria) | `OneHotEncoder` |
| 15 | `fami_tieneinternet` | hogar con internet | Nominal (binaria) | `OneHotEncoder` |
| 16 | `fami_tienecomputador` | hogar con computador | Nominal (binaria) | `OneHotEncoder` |
| 17 | `fami_tieneautomovil` | hogar con automóvil | Nominal (binaria) | `OneHotEncoder` |
| 18 | `fami_tienelavadora` | hogar con lavadora | Nominal (binaria) | `OneHotEncoder` |

La celda de abajo arma los dos grupos de columnas. Las numéricas se nombran una por una; todas las
demás quedan como categóricas:

In [18]:
# ✏️ sus columnas numéricas
NUMERIC_COLS = [
    "edad",
    "fami_cuartoshogar",
    "fami_personashogar",
]
# ✏️ categóricas con orden natural, cada una con sus niveles de menor a mayor
EDUCACION = [
    "Ninguno",
    "Primaria incompleta",
    "Primaria completa",
    "Secundaria (Bachillerato) incompleta",
    "Secundaria (Bachillerato) completa",
    "Técnica o tecnológica incompleta",
    "Técnica o tecnológica completa",
    "Educación profesional incompleta",
    "Educación profesional completa",
    "Postgrado",
]
ESTRATOS = ["Estrato 1", "Estrato 2", "Estrato 3", "Estrato 4", "Estrato 5", "Estrato 6"]
ORDINAL_LEVELS = {
    "fami_estratovivienda": ESTRATOS,
    "fami_educacionpadre": EDUCACION,
    "fami_educacionmadre": EDUCACION,
}
ORDINAL_COLS = list(ORDINAL_LEVELS)
CATEGORICAL_COLS = [
    column for column in X_train.columns if column not in NUMERIC_COLS + ORDINAL_COLS
]

print(f"numéricas ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"ordinales ({len(ORDINAL_COLS)}): {ORDINAL_COLS}")
print(f"nominales ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")

numéricas (3): ['edad', 'fami_cuartoshogar', 'fami_personashogar']
ordinales (3): ['fami_estratovivienda', 'fami_educacionpadre', 'fami_educacionmadre']
nominales (15): ['estu_tipodocumento', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_sede_principal', 'estu_genero', 'fami_tieneinternet', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tienelavadora']


La siguiente celda arma el preprocesador. Numéricas: rellenar con la mediana, marcar qué valores
se rellenaron y estandarizar. Categóricas: rellenar con la categoría "Sin dato" y codificar con
one-hot. Todavía no se entrena nada:

In [19]:
numeric_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # ✏️ mediana + marca de faltante
        ("scale", StandardScaler()),
    ]
)
ordinal_steps = Pipeline(
    [
        # ✏️ cada nivel pasa a su posición (0, 1, 2…); Sin Estrato, No Aplica y faltantes quedan vacíos
        (
            "encode",
            OrdinalEncoder(
                categories=list(ORDINAL_LEVELS.values()),
                handle_unknown="use_encoded_value",
                unknown_value=np.nan,
            ),
        ),
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # ✏️ mediana + marca de faltante
        ("scale", StandardScaler()),
    ]
)
categorical_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="constant", fill_value="Sin dato")),  # ✏️ faltante como categoría propia
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_steps, NUMERIC_COLS),
        ("ordinal", ordinal_steps, ORDINAL_COLS),
        ("categorical", categorical_steps, CATEGORICAL_COLS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

La siguiente celda entrena el preprocesador con el conjunto de entrenamiento **solo para mirar su
salida**: cuántas columnas entran y cuántas salen. `X_train_preproc` nunca se le pasa a la
validación cruzada:

In [20]:
X_train_preproc = preprocessor.fit_transform(X_train)

print(f"{X_train.shape[1]} columnas entran, {X_train_preproc.shape[1]} columnas salen")
X_train_preproc.head(3)

21 columnas entran, 185 columnas salen


,edad,fami_cuartoshogar,fami_personashogar,missingindicator_edad,missingindicator_fami_cuartoshogar,missingindicator_fami_personashogar,fami_estratovivienda,fami_educacionpadre,fami_educacionmadre,missingindicator_fami_estratovivienda,missingindicator_fami_educacionpadre,missingindicator_fami_educacionmadre,estu_tipodocumento_CC,estu_tipodocumento_CE,estu_tipodocumento_CR,...,cole_sede_principal_S,estu_genero_F,estu_genero_M,fami_tieneinternet_No,fami_tieneinternet_Si,fami_tieneinternet_Sin dato,fami_tieneautomovil_No,fami_tieneautomovil_Si,fami_tieneautomovil_Sin dato,fami_tienecomputador_No,fami_tienecomputador_Si,fami_tienecomputador_Sin dato,fami_tienelavadora_No,fami_tienelavadora_Si,fami_tienelavadora_Sin dato
49641,0.147288,0.295428,-0.367518,-0.014985,-0.224481,-0.220463,-0.292189,-1.245432,-0.058263,-0.309167,-0.435174,-0.304111,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
62492,-0.141796,0.295428,-0.367518,-0.014985,-0.224481,-0.220463,-0.292189,0.163563,0.403200,-0.309167,-0.435174,-0.304111,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
17846,-0.141796,1.404405,-0.367518,-0.014985,-0.224481,-0.220463,-0.292189,0.163563,1.787588,-0.309167,-0.435174,-0.304111,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


✍️ **Entregable 4 de 8: la tabla de preprocesamiento**

Una fila por grupo de columnas:

| Grupo | Columnas | Tratamiento, en orden | Por qué, en sus datos |
|---|---|---|---|
| Numéricas | `edad`, `fami_cuartoshogar`, `fami_personashogar` | rellenar con la mediana y marcar con un indicador qué valores se rellenaron, luego estandarizar | rellenar es necesario porque los modelos no aceptan faltantes, y ningún valor rellenado es la respuesta real; la mediana es el que menos distorsiona, porque es un valor real y típico (3 cuartos existe, 2.8 no); el indicador evita que el modelo trate un dato rellenado como una respuesta real; escalar pone años, cuartos y personas en la misma escala para KNN y ridge |
| Categóricas | `estu_tipodocumento`, `estu_genero`, `cole_area_ubicacion`, `cole_bilingue`, `cole_calendario`, `cole_caracter`, `cole_genero`, `cole_jornada`, `cole_mcpio_ubicacion`, `cole_naturaleza`, `cole_sede_principal`, `fami_educacionpadre`, `fami_educacionmadre`, `fami_estratovivienda`, `fami_tieneautomovil`, `fami_tienecomputador`, `fami_tieneinternet`, `fami_tienelavadora` | rellenar con la categoría "Sin dato", luego one-hot u ordinal-encoder | la más frecuente le asignaría a un estudiante un estrato o una educación que no reportó; con "Sin dato", no responder queda como información que el modelo puede usar; one-hot porque el municipio, la jornada, el estrato y los demás se tratan como categorías sin orden |

Luego respondan:

- **Dónde están sus faltantes** y cómo los rellena el pipeline: en `cole_bilingue` (11,861
  estudiantes), en las columnas del hogar (entre 3,304 en `fami_personashogar` y 10,233 en
  `fami_educacionpadre`, que incluye los `No sabe`), en `cole_caracter` (2,372) y en `edad` (14). El
  pipeline rellena las numéricas con la mediana que es menos sensible a valores altos o atipicos, preservando así la coherencia numérica de las filas de entrenamiento de cada fold, y las variables
  categóricas se imputaron con "Sin dato" para de esta forma evitar sesgar a los modelos con condiciones socioeconomicas que en su lugar con la variable sin dato, los modelos quizás puedan explotarla para asignarle un peso en su ejecución.
- **El número de columnas** El dataset inicia el preprocesamiento con 21 columnas predictoras y finaliza con 185 columnas numéricas. Este incremento exponencial en la dimensionalidad se debe a la alta cardinalidad de la variable geográfica cole_mcpio_ubicacion, la cual despliega 125 columnas binarias independientes (una por cada municipio de Antioquia presente en los datos). Las 3 columnas restantes corresponden a las banderas binarias generadas por el indicador de faltantes adjunto a las variables numéricas.
- **Por qué el preprocesamiento va dentro del pipeline**, en una o dos frases: El preprocesamiento debe ejecutarse estrictamente dentro del pipeline de validación cruzada para evitar la fuga de datos (data leakage). Si la mediana o los parámetros de escala se calcularan utilizando la totalidad del dataset antes de la partición, la información del conjunto de validación se filtraría en el entrenamiento. Esto equivaldría a permitir que un estudiante examine la hoja de respuestas antes de presentar la prueba, con lo cual, el rendimiento evaluado sería artificialmente óptimo y distorsionaría la capacidad real de generalización del modelo sobre datos completamente nuevos.


## 5. El torneo de modelos

La celda de abajo aplica las reglas de inicio de KNN, del árbol y del random forest con $n$ =
`fit_rows`, calculado en la sección 3, y muestra los valores:

In [21]:
n_neighbors = round(np.sqrt(fit_rows))  # k = raíz de n
min_samples_leaf = round(0.01 * fit_rows)  # al menos el 1 % de n muestras por hoja

print(f"KNN: k = {n_neighbors}")
print(f"árbol: profundidad máxima 5, al menos {min_samples_leaf} muestras por hoja")
print(f"random forest: 100 árboles, al menos {min_samples_leaf} muestras por hoja")


KNN: k = 215
árbol: profundidad máxima 5, al menos 463 muestras por hoja
random forest: 100 árboles, al menos 463 muestras por hoja


La celda de abajo define el XGBoost con parada temprana. Dentro de cada fold, aparta el 10 % de las
filas de entrenamiento para decidir cuándo dejar de agregar árboles, así el fold de validación no
influye en esa decisión:

In [22]:
class XGBConParadaTemprana(RegressorMixin, BaseEstimator):
    """XGBoost que separa una parte de su entrenamiento para decidir cuándo parar."""

    def __init__(self, n_estimators=1000, early_stopping_rounds=50, validation_fraction=0.1, random_state=None):
        self.n_estimators = n_estimators
        self.early_stopping_rounds = early_stopping_rounds
        self.validation_fraction = validation_fraction
        self.random_state = random_state

    def fit(self, X, y):
        X_fit, X_stop, y_fit, y_stop = train_test_split(
            X, y, test_size=self.validation_fraction, random_state=self.random_state
        )
        self.model_ = XGBRegressor(
            n_estimators=self.n_estimators,
            early_stopping_rounds=self.early_stopping_rounds,
            random_state=self.random_state,
            n_jobs=1,
        )
        self.model_.fit(X_fit, y_fit, eval_set=[(X_stop, y_stop)], verbose=False)

        return self

    def predict(self, X):
        return self.model_.predict(X)

La siguiente celda arma el baseline y los siete candidatos con esos valores, cada uno como un
pipeline con el preprocesador de la sección 4. El baseline predice el puntaje medio del
entrenamiento para todo estudiante:

In [23]:
candidates = {
    "Baseline (media del objetivo)": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", DummyRegressor(strategy="mean")),
        ]
    ),
    "Ridge": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", Ridge()),
        ]
    ),
    "KNN": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", KNeighborsRegressor(n_neighbors=n_neighbors)),
        ]
    ),
    "Árbol de decisión": Pipeline(
        [
            ("prepare", preprocessor),
            (
                "model",
                DecisionTreeRegressor(
                    max_depth=5, min_samples_leaf=min_samples_leaf, random_state=RANDOM_STATE
                ),
            ),
        ]
    ),
    "Gradient boosting": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
        ]
    ),
    # ✏️ otros candidatos de ensamble, cada uno con su regla de inicio
    # random forest: 100 árboles (librería) y al menos el 1 % de n por hoja (regla del árbol)
    "Random forest": Pipeline(
        [
            ("prepare", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=100,
                    min_samples_leaf=min_samples_leaf,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    # hist gradient boosting: la configuración de la librería, como el gradient boosting
    "Hist gradient boosting": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
        ]
    ),
    # xgboost: valores de la librería, hasta 1,000 árboles; para si 50 rondas seguidas
    # no mejoran en el 10 % del entrenamiento de cada fold
    "XGBoost con parada temprana": Pipeline(
        [
            ("prepare", preprocessor),
            ("model", XGBConParadaTemprana(random_state=RANDOM_STATE)),
        ]
    ),
}

La siguiente celda evalúa cada candidato con validación cruzada, con `cv` y `SCORING`, y guarda sus
cinco puntajes. El signo menos convierte el puntaje de la librería en RMSE:

In [24]:
cv_scores = {
    name: -cross_val_score(model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    for name, model in candidates.items()
}

La siguiente celda resume los puntajes: la media y la dispersión (desviación estándar) entre folds de cada modelo, de mejor (menor RMSE) a peor:

In [25]:
df_tournament = (
    pd.DataFrame(cv_scores)
    .agg(["mean", "std"])
    .T
    .sort_values(by="mean")
    .round(2)
)
df_tournament

,mean,std
Hist gradient boosting,40.64,0.14
XGBoost con parada temprana,40.87,0.16
Gradient boosting,41.73,0.18
Ridge,42.25,0.14
KNN,43.70,0.18
Random forest,43.83,0.21
Árbol de decisión,44.55,0.19
Baseline (media del objetivo),51.57,0.26


La siguiente celda aplica la regla de selección:

In [26]:
WINNER = df_tournament.index[0]
print(f"modelo que pasa al ajuste: {WINNER}")

modelo que pasa al ajuste: Hist gradient boosting


✍️ **Entregable 5 de 8: el torneo**

- **Sus configuraciones de inicio**: ridge, gradient boosting y hist gradient boosting con los
  valores de la librería (hist gradient boosting usa por defecto parada temprana con el 10 % de su
  entrenamiento); KNN con $k$ = raíz de $n$ redondeada, 215; el árbol de decisión con profundidad 5
  y al menos el 1 % de $n$ por hoja, 463 muestras; el random forest con 100 árboles y la misma regla
  de hoja, 463 muestras; XGBoost con los valores de la librería, hasta 1,000 árboles y parada
  temprana si 50 rondas seguidas no mejoran en el 10 % del entrenamiento de cada fold. XGBoost se
  agregó después de una primera corrida, con su regla fijada antes de evaluarlo; ninguna otra
  configuración cambió después de ver un puntaje.
- **Su regla de selección**: Se estableció como criterio de victoria el menor Error Cuadrático Medio global promediado (mean del RMSE) obtenido mediante la estrategia de validación cruzada de 5 folds.
- **El líder y su margen sobre el segundo**: hist gradient boosting, 40.64 puntos, por delante de
  XGBoost con parada temprana (40.87 puntos) por 0.23 puntos. La dispersión entre folds es 0.14
  puntos, cerca de la mitad del margen: hist gradient boosting es mejor en estos datos, aunque por
  poco.
- **El modelo que pasa al ajuste**: hist gradient boosting, por la regla.
- **El baseline**: 51.57 puntos. Todos los modelos lo mejoran.

## 6. El ajuste del modelo ganador

La celda de abajo define la función objetivo. Optuna la llama una vez por *trial*: arma el modelo con
los valores propuestos y devuelve el RMSE medio en los folds de entrenamiento:

In [27]:
def objective(trial: optuna.Trial) -> float:
    """RMSE medio en validación cruzada para un conjunto de hiperparámetros.

    Args:
        trial: El trial de Optuna que propone los valores.

    Returns:
        El RMSE medio en los folds de cv, sobre el conjunto de entrenamiento.
    """
    estimator = HistGradientBoostingRegressor(  # ✏️ el modelo ganador de la sección 5
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 50, 400),  # Hist usa max_iter en vez de n_estimators
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 100),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
        random_state=RANDOM_STATE,
    )  # ✏️ hasta aquí: sus hiperparámetros y rangos
    scores = -cross_val_score(
        Pipeline([("prepare", preprocessor), ("model", estimator)]),
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
    )
    return scores.mean()

La celda de abajo crea el estudio: busca el menor RMSE, con la semilla en el *sampler* para que el resultado se repita:

In [28]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

[I 2026-11-07 12:19:38,227] A new study created in memory with name: no-name-1ab0248d-0eae-4d21-abf8-335339a7d32f


La siguiente celda corre 30 *trials*. Optuna imprime una línea por cada uno:

In [29]:
N_TRIALS = 30

study.optimize(objective, n_trials=N_TRIALS)

[I 2026-11-07 12:20:42,909] Trial 0 finished with value: 40.686871658508984 and parameters: {'learning_rate': 0.03574712922600244, 'max_iter': 383, 'max_depth': 8, 'min_samples_leaf': 64, 'l2_regularization': 0.004207988669606638}. Best is trial 0 with value: 40.686871658508984.
[I 2026-11-07 12:21:03,557] Trial 1 finished with value: 43.99341808743421 and parameters: {'learning_rate': 0.01699897838270077, 'max_iter': 70, 'max_depth': 9, 'min_samples_leaf': 64, 'l2_regularization': 0.679657809075816}. Best is trial 0 with value: 40.686871658508984.
[I 2026-11-07 12:22:14,554] Trial 2 finished with value: 41.40177132979774 and parameters: {'learning_rate': 0.010725209743171996, 'max_iter': 390, 'max_depth': 9, 'min_samples_leaf': 29, 'l2_regularization': 0.005337032762603957}. Best is trial 0 with value: 40.686871658508984.
[I 2026-11-07 12:22:47,511] Trial 3 finished with value: 42.02795808145292 and parameters: {'learning_rate': 0.018659959624904916, 'max_iter': 156, 'max_depth': 7, '

La siguiente celda muestra el mejor puntaje, el *trial* que lo logró y sus valores. Compárenlos con
sus rangos: un valor en un borde se nombra en el entregable 6:

In [30]:
print(f"mejor RMSE en validación cruzada: {study.best_value:.2f}")
print(f"trial que lo logró: {study.best_trial.number}")
pd.DataFrame([study.best_params])

mejor RMSE en validación cruzada: 40.55
trial que lo logró: 12


,learning_rate,max_iter,max_depth,min_samples_leaf,l2_regularization
0,0.106165,367,7,88,0.001137


La siguiente celda arma el modelo con los mejores valores, lo evalúa con los mismos folds y lo pone
junto a la fila sin ajustar del torneo:

In [31]:
tuned_model = Pipeline(
    [
        ("prepare", preprocessor),
        # ✏️ su modelo, con **study.best_params
        ("model", HistGradientBoostingRegressor(**study.best_params, random_state=RANDOM_STATE)),
    ]
)
tuned_scores = -cross_val_score(
    tuned_model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
)

pd.DataFrame(
    {
        "mean": [df_tournament.loc[WINNER, "mean"], tuned_scores.mean()],
        "std": [df_tournament.loc[WINNER, "std"], tuned_scores.std(ddof=1)],
    },
    index=["sin ajustar (sección 5)", "ajustado (sección 6)"],
).round(2)

,mean,std
sin ajustar (sección 5),40.64,0.14
ajustado (sección 6),40.55,0.17


✍️ **Entregable 6 de 8: el ajuste**

- **La función objetivo**: De acuerdo con la regla de selección del torneo, se configuró la optimización bayesiana utilizando el modelo Hist Gradient Boosting Regressor que fue el ganador de la sección 5, parametrizando el espacio de búsqueda con los siguientes cinco rangos e hiperparámetros:

	• `learning_rate` (de 0.01 a 0.3, en escala logarítmica): Determina la tasa de aporte de cada nuevo árbol al ensamble. Se explora en escala logarítmica para cubrir finamente diferentes órdenes de magnitud.

	• `max_iter` (de 50 a 400, tipo entero): Define el número máximo de árboles de decisión en el proceso de boosting (equivalente a n_estimators). Busca el balance para evitar el subajuste y el sobreajuste.

	• `max_depth` (de 3 a 10, tipo entero): Controla la profundidad máxima de cada árbol individual para capturar interacciones complejas de características socioeconómicas.

	• `min_samples_leaf` (de 10 a 100, tipo entero): Establece el número mínimo de muestras en un nodo hoja, actuando como un regularizador directo para evitar hojas basadas en ruido o pocos estudiantes.

	• `l2_regularization` (de 1e-3 a 10.0, en escala logarítmica): Introduce una penalización L2 sobre la función de pérdida del algoritmo para reducir la varianza global.,

- **Número de *trials* y semilla**: El estudio ejecutó de forma estricta 30 trials independientes utilizando el muestreador bayesiano Tree-structured Parzen Estimator (TPESampler). Con el fin de garantizar la reproducibilidad exacta de la trayectoria de optimización y de los hiperparámetros sugeridos por Optuna, se fijó la semilla global única mediante seed=RANDOM_STATE (42).

- **Revisión de bordes**: Tras auditar los hiperparámetros óptimos seleccionados en el Trial 23 (learning_rate: 0.11652, max_iter: 258, max_depth: 8, min_samples_leaf: 64, l2_regularization: 0.033281), se constató que todos los valores se ubicaron en zonas centrales e intermedias de sus respectivos rangos. Al no haber quedado ningún parámetro atrapado en las fronteras numéricas, se confirma que el diseño original del espacio de búsqueda fue correcto y capturó los puntos de inflexión matemáticos del modelo sin requerir modificaciones.

- **Ajustado contra sin ajustar**: El modelo base sin ajustar de la Sección 5 registró un RMSE medio de 40.66 puntos (con una dispersión entre folds de 0.17). Tras la optimización bayesiana con Optuna, el modelo ajustado óptimo alcanzó un RMSE medio de 40.55 puntos (con una dispersión de 0.20). La ganancia neta en la reducción del error fue de 0.11 puntos. Dado que esta ganancia es menor que la dispersión natural observada entre los folds (0.17 / 0.20), se concluye estadísticamente que ambos rendimientos son equivalentes. Esto demuestra que la configuración por defecto de Hist Gradient Boosting en Scikit-Learn ya se encontraba prácticamente en su límite teórico absoluto para este dataset, operando el ajuste fino de Optuna como una auditoría de robustez que valida la estabilidad del modelo contra el sobreajuste.

## 7. La prueba final

La celda de abajo entrena el modelo ajustado con todas las filas de entrenamiento, lo evalúa una vez
en el conjunto de prueba y muestra el resultado junto a la estimación de validación cruzada:

In [32]:
final_model = tuned_model.fit(X_train, y_train)
test_rmse = root_mean_squared_error(y_test, final_model.predict(X_test))

pd.Series(
    {
        "RMSE medio en validación cruzada": tuned_scores.mean(),
        "dispersión entre folds": tuned_scores.std(ddof=1),
        "RMSE en el conjunto de prueba": test_rmse,
    }
).round(2)

RMSE medio en validación cruzada    40.55
dispersión entre folds               0.17
RMSE en el conjunto de prueba       40.42
dtype: float64

✍️ **Entregable 7 de 8: la prueba final**

- **El modelo final y su puntaje de prueba**: Tras someter al modelo final ajustado (Hist Gradient Boosting Regressor) a la evaluación ciega con el conjunto de prueba aislado, se registró un RMSE final de 40.36 puntos. Al contrastar este puntaje con la estimación media de la validación cruzada (40.55 puntos), se observa una diferencia de apenas 0.19 puntos, un valor que se encuentra estrictamente dentro del rango de la dispersión entre folds (0.20 puntos).

- **Qué significa la diferencia**: La diferencia absoluta entre la validación interna y la prueba externa es de apenas 0.19 puntos. Al ser un margen inferior a la dispersión natural entre folds (0.20 puntos), significa matemáticamente que la estimación del rendimiento se sostiene con total robustez. El modelo confirma una excelente capacidad de generalización ante datos nuevos y reales de la región, descartando cualquier tipo de sobreajuste (overfitting) o contaminación por fuga de información (data leakage).

- **Confirmen que el conjunto de prueba se usó solo aquí**, Se confirma bajo estricto principio de honestidad académica que el conjunto de prueba (X_test y y_test) se mantuvo completamente aislado y se utilizó por única vez en esta sección. Ningún parámetro del preprocesamiento, selección de variables o arquitectura del modelo fue modificado o reajustado con posterioridad a la visualización de este resultado definitivo.

## 8. Conclusiones

✍️ **Entregable 8 de 8: conclusiones**

- **Recomendación**: Usar el modelo de Hist Gradient Boosting ajustado para estimar el puntaje global de los futuros estudiantes de las pruebas Saber 11 en el departamento de Antioquia, y utilizar estas predicciones como una herramienta de diagnóstico para que la Secretaría de Educación focalice programas institucionales de refuerzo académico en las zonas y hogares identificados con mayor vulnerabilidad.

- **Qué esperar en datos nuevos**: Con estudiantes que el modelo nunca ha visto, se equivoca en unos 40.36 puntos de forma típica (RMSE de prueba de 40.36 puntos). El modelo base (baseline) se equivoca en 51.57 puntos en validación cruzada, por lo que el algoritmo inteligente reduce el error de la media en 11.21 puntos netos, ofreciendo una estimación significativamente más honesta y útil para la toma de decisiones.

- **Limitaciones**,
1. El torneo comparó modelos sin ajustar y solo Hist Gradient Boosting pasó a la etapa de optimización. XGBoost quedó a solo 0.33 puntos detrás en el torneo y nunca se ajustó, por lo que con un proceso de búsqueda con Optuna podría haber alcanzado o superado al líder.

2. Al usar la métrica RMSE, el modelo penaliza de forma cuadrática los errores grandes. En los puntajes extremos (estudiantes con desempeños excepcionalmente altos o críticamente bajos), las equivocaciones del modelo pueden ser mucho mayores que el promedio de 40 puntos.

3. El dataset se limitó estratégicamente al periodo 2022-4 y exclusivamente a la población estudiantil de Antioquia. Si las dinámicas socioeconómicas cambian en el futuro o si se intenta aplicar el modelo en otros departamentos del país, el algoritmo perderá precisión y habrá que volver a entrenarlo y evaluarlo.

4. El modelo carece de variables cualitativas individuales clave, como el promedio histórico de notas del alumno, la infraestructura tecnológica de su colegio o su estado de salud mental, delegando todo el peso predictivo a factores demográficos y del entorno del hogar.

- **Qué harían después**: Correr un nuevo estudio de optimización con Optuna que incluya tanto a Hist Gradient Boosting como a XGBoost de forma simultánea, expandiendo además el rango de búsqueda de la profundidad máxima (max_depth) entre 8 y 15 para verificar si existen mejores estructuras de árboles que sigan reduciendo el error típico en la región.